# Calendar Dimension

Builds the standard calendar table `hcp_databricks_workspace.silver.calender` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
CREATE OR REPLACE TABLE hcp_databricks_workspace.silver.calender
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  CAST(date_format(date, 'yyyyMMdd') AS INT)                AS datekey,
  date,
  -- Day
  day(date)                                                 AS day_of_month,
  weekday(date) + 1                                         AS day_of_week,          -- 1 = Monday ... 7 = Sunday (ISO)
  date_format(date, 'EEEE')                                 AS day_name,
  date_format(date, 'EEE')                                  AS day_short_name,
  dayofyear(date)                                           AS day_of_year,
  weekday(date) >= 5                                        AS is_weekend,
  weekday(date) < 5                                         AS is_weekday,
  -- Week (ISO, weeks start Monday)
  weekofyear(date)                                          AS week_of_year,
  extract(YEAROFWEEK FROM date)                             AS iso_week_year,
  CAST(ceil(day(date) / 7.0) AS INT)                        AS week_of_month,
  CAST(date_trunc('WEEK', date) AS DATE)                    AS week_start_date,
  date_add(CAST(date_trunc('WEEK', date) AS DATE), 6)       AS week_end_date,
  -- Month
  month(date)                                               AS month,
  date_format(date, 'MMMM')                                 AS month_name,
  date_format(date, 'MMM')                                  AS month_short_name,
  CAST(date_format(date, 'yyyyMM') AS INT)                  AS year_month,
  date_format(date, 'MMM-yyyy')                             AS month_year,
  CAST(date_trunc('MONTH', date) AS DATE)                   AS first_day_of_month,
  last_day(date)                                            AS last_day_of_month,
  day(last_day(date))                                       AS days_in_month,
  date = CAST(date_trunc('MONTH', date) AS DATE)            AS is_month_start,
  date = last_day(date)                                     AS is_month_end,
  -- Quarter
  quarter(date)                                             AS quarter,
  concat('Q', quarter(date))                                AS quarter_name,
  concat(year(date), '-Q', quarter(date))                   AS year_quarter,
  CAST(date_trunc('QUARTER', date) AS DATE)                 AS first_day_of_quarter,
  date_sub(add_months(CAST(date_trunc('QUARTER', date) AS DATE), 3), 1) AS last_day_of_quarter,
  datediff(date, CAST(date_trunc('QUARTER', date) AS DATE)) + 1         AS day_of_quarter,
  -- Half year
  IF(month(date) <= 6, 1, 2)                                AS half_year,
  concat('H', IF(month(date) <= 6, 1, 2))                   AS half_year_name,
  -- Year
  year(date)                                                AS year,
  CAST(date_trunc('YEAR', date) AS DATE)                    AS first_day_of_year,
  make_date(year(date), 12, 31)                             AS last_day_of_year,
  day(last_day(make_date(year(date), 2, 1))) = 29           AS is_leap_year,
  IF(day(last_day(make_date(year(date), 2, 1))) = 29, 366, 365) AS days_in_year,
  CAST(floor(year(date) / 10) * 10 AS INT)                  AS decade
FROM dates
ORDER BY date

In [0]:
SELECT
  count(*)                    AS row_count,
  count(DISTINCT datekey)     AS distinct_datekeys,
  min(date)                   AS min_date,
  max(date)                   AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_rows,
  sum(CAST(is_leap_year AND month = 2 AND day_of_month = 29 AS INT)) AS leap_days
FROM hcp_databricks_workspace.silver.calender

In [0]:
SELECT *
FROM hcp_databricks_workspace.silver.calender
WHERE date BETWEEN DATE'2024-12-28' AND DATE'2025-01-03'
ORDER BY date